In [4]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain_deepseek import ChatDeepSeek

from dotenv import load_dotenv
load_dotenv(override=True)

#0. 连接LLM模型
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

#1. 定义状态
class OverAllState(MessagesState):
    username: str
    output: str

#2. 定义节点
def node_a(state: OverAllState) -> OverAllState:
    return {
        "messages": [HumanMessage("你好,我是" + state["username"])],
    }

def llm_node(state: OverAllState) -> OverAllState:
    res = model.invoke(state["messages"])
    return {
        "messages": [res],
        "output": res.content
    }

#3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()

#4. 运行图
result = graph.invoke({"username": "老王"})
print(result)


{'messages': [HumanMessage(content='你好,我是老王', additional_kwargs={}, response_metadata={}, id='e487a0e6-476d-439b-aef0-6f41e9637b6d'), AIMessage(content='你好，老王！很高兴认识你。今天有什么可以帮你的吗？无论是聊天、咨询问题，还是需要建议，我都在这儿呢。😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 8, 'total_tokens': 39, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '3bae9463-2d76-46f0-b26d-ac2590c60b5f', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f17f0-e16b-7e93-bd71-ca47c5266c35-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 31, 'total_tokens': 39, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})], 'username': '老王', 'output'